In [1]:
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
import pandas as pd
import h5py
from typing import Literal
import struct

plt.rcParams["figure.dpi"] = 256

In [2]:
def plot_with_peaks(
    data,
    n_bins,
    prominence=None,
    xlabel="",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        plt.bar(bin_centers, counts, width=bin_width, align="center", bottom=bottom)
        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.03f} ± {half_width:.03f} MeV",
                    (x, y + height_scale * (0.005 * text_size)),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel("Count")
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    # plt.figure()
    plt.show()

# Loaders

In [3]:
keys = [
    "E",
    "name",
    "z",
    "rho",
    "phi",
    "momentumZ",
    "momentumRho",
    "momentumPhi",
    "sourceDepth",
    "sourceRadius",
    "sourceE",
    "sourcePDG"
]

def decode(b):
    return b.decode("utf-8")


decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["Measurement"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} interactions recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [4]:
df = load_csv("archive/transport.csv")
df

27,762 interactions recorded


,E,name,z,rho,phi,momentumZ,momentumRho,momentumPhi,sourceDepth,sourceRadius,sourceE,sourcePDG
0,0.142231,gamma,-1.776360e-15,1603.7700,-2.69755,0.239647,0.970860,0.432922,-73.49670,1783.500,0.393452,22
1,0.220309,gamma,0.000000e+00,214.1180,-1.77846,0.954033,0.299701,1.492680,-12.73120,314.460,0.475115,11
2,0.160819,gamma,0.000000e+00,268.1030,2.00307,0.692368,0.721545,-1.041880,-167.97800,757.633,0.510999,-11
3,0.450666,gamma,0.000000e+00,853.8700,-2.22400,0.362090,0.932143,0.871855,-6.92746,952.331,0.510999,-11
4,0.128852,gamma,0.000000e+00,202.9250,-2.73611,0.592401,0.805644,0.501674,-346.16600,389.364,2.223040,2112
...,...,...,...,...,...,...,...,...,...,...,...,...
27757,0.460941,gamma,0.000000e+00,114.3780,2.26937,0.862319,0.506365,-1.071570,-126.38900,167.796,0.510999,22
27758,0.510999,gamma,-3.552710e-15,127.6900,-2.53552,0.853898,0.520440,1.213030,-55.61480,156.729,0.510999,22
27759,0.510999,gamma,0.000000e+00,1420.3600,2.62118,0.226954,0.973906,-0.560565,-21.43790,1512.280,0.510999,-11
27760,0.322524,gamma,3.552710e-15,1145.3800,2.76216,0.352770,0.935710,-0.339877,-6.73789,1295.640,0.641208,22


# Analysis

In [5]:
mask = df["z"].between(200, 368)
df[mask]

,E,name,z,rho,phi,momentumZ,momentumRho,momentumPhi,sourceDepth,sourceRadius,sourceE,sourcePDG
184,0.338476,gamma,304.110,1731.8400,-2.234080,-0.037972,0.999279,0.935697,-104.9890,2141.050,0.510999,22
270,1.199160,e-,213.633,85.3790,2.422420,0.330879,0.943673,-0.230233,-186.7340,156.706,5.601850,-13
348,0.131971,gamma,226.898,950.8940,-0.712273,0.012276,0.999925,2.446390,-162.6980,1196.760,0.510999,22
859,0.114234,gamma,272.764,228.2290,0.782586,0.300712,0.953715,-2.256240,-52.0495,1455.650,0.510999,22
1114,2.916060,e-,210.781,83.4419,0.640601,0.493331,0.869842,-2.790430,-268.8570,538.968,7.106330,2112
...,...,...,...,...,...,...,...,...,...,...,...,...
27379,0.240421,gamma,243.010,1115.3000,-1.086640,-0.076975,0.997033,2.086450,-48.1526,1655.800,12.388100,22
27490,1.211000,e-,358.956,83.1717,2.435780,0.869393,0.494122,0.530486,-217.6310,325.839,4.361330,22
27567,0.137030,gamma,354.241,478.1740,-2.563770,-0.061132,0.998130,0.644613,-65.7644,627.595,19.650600,11
27573,0.427529,gamma,211.237,84.9262,-3.051730,0.113653,0.993520,-1.098350,-15.6085,304.622,0.510999,22


## Masking Utilities

In [6]:
property_range = lambda property, lower, upper, mask=None: (
    df if mask is None else df[mask]
)[property].between(lower, upper)
energy_range = lambda lower, upper, mask=None: property_range("E", lower, upper, mask)

In [7]:
# plot_with_peaks(df["sourceDepth"], 100)
# plot_with_peaks(df["sourceRadius"], 100)
# plt.yscale("log")
# plot_with_peaks(df["z"], 100)
# plot_with_peaks(df["rho"], 100)
# plot_with_peaks(df["phi"], 100)
# plot_with_peaks(df["momentumZ"], 100)
# plot_with_peaks(df["momentumRho"], 100)
diff = (df["phi"]-df["momentumPhi"]) % (2 * np.pi) - np.pi
plot_with_peaks(diff, 100)
df[mask]

,E,name,z,rho,phi,momentumZ,momentumRho,momentumPhi,sourceDepth,sourceRadius,sourceE,sourcePDG
184,0.338476,gamma,304.110,1731.8400,-2.234080,-0.037972,0.999279,0.935697,-104.9890,2141.050,0.510999,22
270,1.199160,e-,213.633,85.3790,2.422420,0.330879,0.943673,-0.230233,-186.7340,156.706,5.601850,-13
348,0.131971,gamma,226.898,950.8940,-0.712273,0.012276,0.999925,2.446390,-162.6980,1196.760,0.510999,22
859,0.114234,gamma,272.764,228.2290,0.782586,0.300712,0.953715,-2.256240,-52.0495,1455.650,0.510999,22
1114,2.916060,e-,210.781,83.4419,0.640601,0.493331,0.869842,-2.790430,-268.8570,538.968,7.106330,2112
...,...,...,...,...,...,...,...,...,...,...,...,...
27379,0.240421,gamma,243.010,1115.3000,-1.086640,-0.076975,0.997033,2.086450,-48.1526,1655.800,12.388100,22
27490,1.211000,e-,358.956,83.1717,2.435780,0.869393,0.494122,0.530486,-217.6310,325.839,4.361330,22
27567,0.137030,gamma,354.241,478.1740,-2.563770,-0.061132,0.998130,0.644613,-65.7644,627.595,19.650600,11
27573,0.427529,gamma,211.237,84.9262,-3.051730,0.113653,0.993520,-1.098350,-15.6085,304.622,0.510999,22


In [8]:
plt.close()
plt.hexbin(
    df["rho"] * np.cos(df["phi"]),
    df["rho"] * np.sin(df["phi"]),
    # C=df["E"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.colorbar(label="Count")
# plt.xlabel("$z$-Momentum Bias, $p_z$")
# plt.ylabel("Energy, $E$ (MeV)")
# plt.title("Number of Gamma Rays Produced Below 10 MeV")
plt.show()

In [9]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

mask = ((df["momentumPhi"] - df["phi"]) % (2 * np.pi)).abs() < np.pi/2
d = df

# ax.quiver(
#     d["rho"] * np.cos(d["phi"]),
#     d["rho"] * np.sin(d["phi"]),
#     d["z"],
#     d["momentumRho"] * np.cos(d["momentumPhi"]),
#     d["momentumRho"] * np.sin(d["momentumPhi"]),
#     d["momentumZ"],
#     length=50,
#     linewidth=0.1,
#     arrow_length_ratio=0.3
# )
scatter_plot = ax.scatter(
    d["rho"] * np.cos(d["phi"]),
    d["rho"] * np.sin(d["phi"]),
    d["z"],
    s=1,
    c=(d["phi"]-d["momentumPhi"]) % (2.0 * np.pi) - np.pi,
)

fig.colorbar(scatter_plot, ax=ax)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()